## \u2b50 Bonus — ask the *entire* JupyterHub docs with RAG

<details>
<summary><b>Click to expand</b></summary>

Customizing a hub means digging through the **Zero-to-JupyterHub** docs. Instead,
the same RAG idea from Part 2 indexes the **whole** docs site, then you just ask.

**Run the (collapsed) build cell once** — it downloads and indexes every docs
page with NRP's managed embedding model (~30 s). Then run the **ask** cell: it
**prompts you** for a question and answers from the docs, citing the page. Re-run
the ask cell as many times as you like — no rebuilding, no editing code.

</details>

## Key concepts

- **Helm** is the package manager for Kubernetes: instead of writing every
  Deployment, Service, and ConfigMap by hand, you install a **chart** (a bundle
  of templates) and tune it through a **values file**.
- The **JupyterHub chart** deploys a *hub*, a *proxy*, and per-user notebook
  servers. You set authentication, images, CPU/RAM/GPU limits, and storage in the
  values file.
- **One hub per namespace** — that's why each of you has your **own** namespace
  (`nrp-training-000` … `nrp-training-099`, on the slip you were handed).
- In this training session, `kubectl`, `helm`, and your kubeconfig are already
  set up — nothing to install.

## 1. Set your namespace and verify access

Put **your** assigned namespace in the cell below and run it. It should print
`yes` — that confirms your service account can deploy a hub there.

In [ ]:
cd ~/cra-tutorial   # the Bash kernel starts in your home dir; work from the repo
# 👇 CHANGE this to YOUR assigned namespace (from your slip), then run the cell.
export NAMESPACE=nrp-training-000

kubectl auth can-i create deployment -n "$NAMESPACE"   # expect: yes

## 2. Add the JupyterHub Helm repository

This tells Helm where to download the chart from.

In [ ]:
helm repo add jupyterhub https://jupyterhub.github.io/helm-chart/
helm repo update
echo "---"
kubectl auth whoami && helm version --short

## 3. Review the values file

This repo ships a complete, **NRP-tested** values file at
[`yamls/jhub-values.yaml`](yamls/jhub-values.yaml). A few settings in it are
non-obvious but **required on NRP** (the stock chart defaults get rejected):

- `service.type: ClusterIP` — NRP blocks `LoadBalancer` services.
- `hub.resources` / `proxy.chp.resources` with **requests *and* limits** — NRP
  requires every container to declare them.
- `scheduling.userScheduler` / `prePuller` **disabled** — they need
  *cluster-scoped* RBAC your namespace account doesn't have.
- `storageClass: rook-ceph-block-east` + region `us-east` — matches the training
  cluster's storage.

The parts you'll most likely tweak are the **password** and the **image
profiles**. Take a look:

In [ ]:
cd ~/cra-tutorial   # the Bash kernel starts in your home dir; work from the repo
cat yamls/jhub-values.yaml

## 4. Deploy with Helm

This installs the chart into **your** namespace with the values file. It waits
until the hub is ready (~1–2 min) and prints `STATUS: deployed`.

In [ ]:
cd ~/cra-tutorial   # the Bash kernel starts in your home dir; work from the repo
helm upgrade --cleanup-on-fail --install jhub jupyterhub/jupyterhub \
  --namespace "$NAMESPACE" \
  --version 3.3.7 \
  --values yamls/jhub-values.yaml \
  --wait --timeout 10m

See what got created — you should see a **hub** pod and a **proxy** pod
reach `Running` (user pods appear on demand when someone logs in):

In [ ]:
kubectl get pods,svc -n "$NAMESPACE"

## 5. Make it reachable at a URL

By default the hub is internal (`ClusterIP`). Add an **ingress** to expose it at a
public address. Pick a unique host name, then upgrade the release in place:

In [ ]:
cd ~/cra-tutorial   # the Bash kernel starts in your home dir; work from the repo
export HUBHOST="jhub-${NAMESPACE}.nrp-nautilus.io"

helm upgrade jhub jupyterhub/jupyterhub \
  --namespace "$NAMESPACE" --version 3.3.7 \
  --values yamls/jhub-values.yaml \
  --set ingress.enabled=true \
  --set ingress.ingressClassName=haproxy \
  --set ingress.hosts[0]="$HUBHOST" \
  --set ingress.tls[0].hosts[0]="$HUBHOST" \
  --wait --timeout 5m

echo "Give DNS + TLS a minute, then log in at:  https://$HUBHOST   (admin / training123)"

## 6. Customize

Customizing = change a value and re-run the upgrade. For small tweaks use
`--set`; for bigger ones edit `yamls/jhub-values.yaml` and re-run the deploy
cell. Example — guarantee every user 4 GB of RAM:

In [ ]:
cd ~/cra-tutorial   # the Bash kernel starts in your home dir; work from the repo
helm upgrade jhub jupyterhub/jupyterhub \
  --namespace "$NAMESPACE" --version 3.3.7 \
  --values yamls/jhub-values.yaml \
  --set ingress.enabled=true \
  --set ingress.ingressClassName=haproxy \
  --set ingress.hosts[0]="$HUBHOST" \
  --set ingress.tls[0].hosts[0]="$HUBHOST" \
  --set singleuser.memory.guarantee=4G \
  --wait --timeout 5m

echo "Updated: per-user RAM guarantee is now 4G"

## 7. Manage and clean up

Check your release and the hub logs:

In [ ]:
helm list -n "$NAMESPACE"
echo "--- hub logs (last 20 lines) ---"
kubectl logs -n "$NAMESPACE" -l component=hub --tail=20

**Please tear it down when you're done** so your namespace is free:

In [ ]:
helm uninstall jhub -n "$NAMESPACE"
# user-data PVCs are kept by default; delete them too only if you're sure:
# kubectl delete pvc -n "$NAMESPACE" -l component=singleuser-storage
kubectl get pods -n "$NAMESPACE"

## ⭐ Bonus — ask the *entire* JupyterHub docs with RAG

<details>
<summary><b>Click to expand</b></summary>

Customizing a hub means digging through the **Zero-to-JupyterHub** docs. Instead,
point the *same RAG idea from Part 2* at the **whole documentation set** and just
ask. The cell below (collapsed — click ▶/the dots to expand) indexes every docs
page with NRP's managed embedding model, then answers a few questions, citing the
page it used. Edit the `QUESTIONS` list to ask your own.

It's a single self-contained `python3` block, so it runs right here under the
Bash kernel.

</details>

In [ ]:
# (Collapsed) Build a searchable index of the ENTIRE JupyterHub docs — run once (~30s).
mkdir -p /tmp/z2jh
python3 - <<'PYEOF'
import os, re, requests, numpy as np
from openai import OpenAI
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
BASE = "https://raw.githubusercontent.com/jupyterhub/zero-to-jupyterhub-k8s/main/docs/source/"
PAGES = [
    "administrator/advanced", "administrator/architecture", "administrator/authentication",
    "administrator/cost", "administrator/debug", "administrator/index",
    "administrator/optimization", "administrator/security", "administrator/services",
    "administrator/troubleshooting", "administrator/upgrading/index",
    "administrator/upgrading/upgrade-1-to-2", "administrator/upgrading/upgrade-2-to-3",
    "administrator/upgrading/upgrade-3-to-4", "index", "jupyterhub/customization",
    "jupyterhub/customizing/extending-jupyterhub",
    "jupyterhub/customizing/user-environment", "jupyterhub/customizing/user-management",
    "jupyterhub/customizing/user-resources", "jupyterhub/customizing/user-storage",
    "jupyterhub/index", "jupyterhub/installation", "jupyterhub/uninstall",
    "kubernetes/amazon/efs_storage", "kubernetes/amazon/step-zero-aws-eks",
    "kubernetes/amazon/step-zero-aws", "kubernetes/digital-ocean/step-zero-digital-ocean",
    "kubernetes/docker-desktop/step-zero-docker-desktop",
    "kubernetes/google/step-zero-gcp", "kubernetes/ibm/step-zero-ibm", "kubernetes/index",
    "kubernetes/microsoft/step-zero-azure", "kubernetes/minikube/step-zero-minikube",
    "kubernetes/other-infrastructure/step-zero-other", "kubernetes/ovh/step-zero-ovh",
    "kubernetes/redhat/step-zero-openshift", "kubernetes/setup-helm",
    "kubernetes/setup-kubernetes", "repo2docker", "resources/community",
    "resources/glossary", "resources/index", "resources/reference-docs", "resources/tools"
]
def _clean(t):
    t = re.sub(r"```\{[^}]*\}", "", t); t = re.sub(r"```.*?```", "", t, flags=re.S)
    t = re.sub(r"\{ref\}`[^`]*`", "", t); t = re.sub(r"\{[a-z]+\}`([^`]*)`", r"\1", t)
    return re.sub(r"\n{3,}", "\n\n", t).strip()
chunks, pages = [], []
for p in PAGES:
    try: txt = _clean(requests.get(BASE + p + ".md", timeout=20).text)
    except Exception: continue
    for i in range(0, len(txt), 550):
        ch = txt[i:i+700]
        if len(ch) > 120: chunks.append(ch); pages.append(p)
vecs = []
for i in range(0, len(chunks), 64):
    r = client.embeddings.create(model="qwen3-embedding", input=chunks[i:i+64])
    vecs += [d.embedding for d in r.data]
vecs = np.array(vecs); vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
np.savez("/tmp/z2jh/index.npz", chunks=np.array(chunks, dtype=object),
         pages=np.array(pages, dtype=object), vecs=vecs)
print(f"Indexed {len(chunks)} chunks from {len(set(pages))} JupyterHub doc pages. Ask away below \U0001f447")
PYEOF

cat > /tmp/z2jh/askdocs.py <<'PYEOF'
import sys, os, numpy as np
from openai import OpenAI
IDX = "/tmp/z2jh/index.npz"
if not os.path.exists(IDX):
    sys.exit("Index not found - run the (collapsed) build cell above first.")
q = " ".join(sys.argv[1:]).strip()
if not q:
    sys.exit('No question given - put one in the QUESTION="..." line and re-run.')
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ["OPENAI_API_BASE"])
d = np.load(IDX, allow_pickle=True)
chunks, pages, vecs = d["chunks"], d["pages"], d["vecs"]
qv = np.array(client.embeddings.create(model="qwen3-embedding", input=[q]).data[0].embedding)
qv /= np.linalg.norm(qv)
order = (vecs @ qv).argsort()[-4:][::-1]
ctx = "\n\n".join(f"[{pages[i]}]\n{chunks[i]}" for i in order)
r = client.chat.completions.create(
    model="minimax-m2", max_tokens=1200, temperature=0.2,
    messages=[{"role": "system", "content": "Answer using ONLY the JupyterHub docs context. "
               "Be concise and show the relevant Helm values YAML when applicable."},
              {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {q}"}])
m = r.choices[0].message
ans = m.content or getattr(m, "reasoning", None) or "(no answer)"
srcs = ", ".join(dict.fromkeys(str(pages[i]) for i in order))
print("\n" + "=" * 78)
print("  QUESTION:", q)
print("=" * 78 + "\n")
print(ans.strip())
print("\n" + "-" * 78)
print("  sources:", srcs)
print("-" * 78)
PYEOF
echo "Ready - ask your question in the next cell."

In [ ]:
# Put your question between the quotes, then run the cell (re-run to ask another):
QUESTION="How do I allow each user to run up to 5 servers?"

python3 /tmp/z2jh/askdocs.py "$QUESTION"